# 05 — Autoencoder Training (Cloud GPU or Local)

Trains the 1D-convolutional autoencoder (`flightops.models.autoencoder`) and compares it with the
Isolation Forest baseline **on the identical split**. Both reports carry a `split_fingerprint`, which
is checked below.

**Runs on:** Google Colab, Kaggle, or locally (the conda env `environ`, CPU).

| Step | What happens |
|------|--------------|
| 1 | Settings |
| 2 | Detect environment and GPU |
| 3 | Get the project code (git clone, or an uploaded `flightops_src.zip`) |
| 4 | Get the dataset: downloaded from NASA or the verified Internet Archive copy and SHA-256 checked. **The data is never re-hosted** (licence listed as "N/A") |
| 5 | Build the float32 working copy |
| 6 | Train the baseline (≈1–2 min), then the autoencoder |
| 7 | Compare results |
| 8 | Package the results to bring back into the local repo |

**Kaggle:** *Settings → Accelerator → GPU T4 x2 (or P100)*, *Settings → Internet → On*, then *Run All*.
**Colab:** *Runtime → Change runtime type → T4 GPU*, then *Run all*.

## 1. Settings

In [ ]:
CODE_SOURCE = "git"          # "git": clone REPO_URL (public repo, push your branch first)
                             # "zip": flightops_src.zip uploaded as a Kaggle dataset / Colab upload
REPO_URL = "https://github.com/sasank-in/FlightOps-AI.git"
BRANCH = "feature/dataset-ingestion"
USE_DRIVE_CACHE = True       # Colab only: keep the 1.7 GB raw download in Google Drive between sessions
RUN_AIRCRAFT_CV = True       # 5 extra trainings with whole aircraft held out (recommended on GPU)
EPOCHS = None                # None = value in configs/model.yaml
SMOKE = False               # True = tiny 2-epoch run to check the pipeline

## 2. Environment and GPU

In [ ]:
import os, sys, subprocess, shutil, zipfile, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/working").exists()
import torch
print("Python", sys.version.split()[0], "| torch", torch.__version__)
print("Environment:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected: training will run on CPU. Fine locally; on Colab/Kaggle enable a GPU runtime.")

## 3. Project code

In [ ]:
def sh(*cmd, cwd=None):
    """Run a command and stream its output into the notebook."""
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(list(map(str, cmd)), cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")

def find_uploaded_code():
    """Kaggle auto-extracts uploaded zips into /kaggle/input/...; Colab keeps the .zip."""
    roots = [Path("/kaggle/input"), Path("/content")]
    for root in filter(Path.exists, roots):
        for p in root.rglob("pyproject.toml"):
            if (p.parent / "src" / "flightops").is_dir():
                return p.parent, None
        for z in root.rglob("flightops_src*.zip"):
            return None, z
    return None, None

if IN_COLAB or IN_KAGGLE:
    # Kaggle: work in /tmp so the ~4.5 GB of data never lands in the notebook's saved output.
    REPO = Path("/content/FlightOps-AI") if IN_COLAB else Path("/tmp/FlightOps-AI")
    # Always refresh the CODE (a kernel restart keeps /tmp, so an old checkout would be reused);
    # downloaded data under REPO/data is untracked/ignored and is kept.
    if CODE_SOURCE == "git":
        if not (REPO / ".git").exists():
            REPO.mkdir(parents=True, exist_ok=True)
            sh("git", "-C", REPO, "init", "-q")
        sh("git", "-C", REPO, "fetch", "--depth", "1", REPO_URL, BRANCH)
        sh("git", "-C", REPO, "reset", "--hard", "FETCH_HEAD")    # latest pushed code; data/ is untouched
    else:
        folder, archive = find_uploaded_code()
        if folder is None and archive is None and IN_COLAB:
            from google.colab import files
            print("Upload flightops_src.zip (created locally with `git archive`, see README)")
            archive = Path("/content") / next(iter(files.upload()))
        shutil.rmtree(REPO / "src", ignore_errors=True)   # drop stale code, keep REPO/data
        if folder is not None:
            shutil.copytree(folder, REPO, dirs_exist_ok=True)   # input is read-only; copy to a writable place
        elif archive is not None:
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(REPO)
        else:
            raise FileNotFoundError("Project code not found: attach the flightops_src dataset, or set CODE_SOURCE = 'git'.")
    sh(sys.executable, "-m", "pip", "install", "-q", "-e", REPO)
else:
    REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

sys.path.insert(0, str(REPO / "src"))   # make the package importable without restarting the kernel
os.environ["FLIGHTOPS_PROJECT_ROOT"] = str(REPO)
os.chdir(REPO)
import importlib, flightops
from flightops.features import build_features
from flightops.models import isolation_forest, autoencoder
for m in (build_features, isolation_forest, autoencoder):
    importlib.reload(m)   # pick up refreshed code even if an older version was imported in this kernel
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"flightops {flightops.__version__} at {REPO} | commit {commit or 'n/a (zip)'}")
print(f"input set {build_features.FEATURE_SET_VERSION} ({len(build_features.channel_names())} channels) | "
      f"Isolation Forest {isolation_forest.MODEL_VERSION} | Autoencoder {autoencoder.MODEL_VERSION}")

## 4. Dataset (download and verify, or reuse a verified copy)

In [ ]:
from flightops.data.download import FILES, download_all, is_valid

RAW = REPO / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)
CACHE = None
if IN_COLAB and USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = Path("/content/drive/MyDrive/flightops-ai-data/raw")
    CACHE.mkdir(parents=True, exist_ok=True)
    for f in FILES:  # copy cached files to fast local disk (memmaps on Drive are slow)
        if (CACHE / f.name).exists() and not (RAW / f.name).exists():
            print("copying from Drive cache:", f.name)
            shutil.copy2(CACHE / f.name, RAW / f.name)

for name, source in download_all(RAW, timeout=30).items():   # verifies size + SHA-256
    print(f"OK {name} <- {source}")

if CACHE is not None:
    for f in FILES:
        if not is_valid(CACHE / f.name, f):
            print("saving to Drive cache:", f.name)
            shutil.copy2(RAW / f.name, CACHE / f.name)

## 5. Float32 working copy

In [ ]:
manifest = REPO / "data" / "processed" / "manifest.json"
if not manifest.exists():
    sh(sys.executable, "-m", "flightops.data.processed", cwd=REPO)
print(json.loads(manifest.read_text())["telemetry"])

## 6. Train: baseline, then autoencoder

In [ ]:
sh(sys.executable, "-W", "ignore", "-m", "flightops.models.train_baseline", cwd=REPO)

In [ ]:
args = ["--smoke"] if SMOKE else []
if RUN_AIRCRAFT_CV and not SMOKE:
    args.append("--aircraft-cv")
if EPOCHS and not SMOKE:
    args += ["--epochs", str(EPOCHS)]
sh(sys.executable, "-W", "ignore", "-m", "flightops.models.train_autoencoder", *args, cwd=REPO)

## 7. Results

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

reports = REPO / "artifacts" / "reports"
ae = json.loads((reports / ("autoencoder_smoke.json" if SMOKE else "autoencoder.json")).read_text())
bl = json.loads((reports / "baseline_isolation_forest.json").read_text())
same_split = ae["split_fingerprint"] == bl["primary_stratified"]["split_fingerprint"]
print("identical split for both models:", same_split, "| AE device:", ae["device"], "| smoke run:", ae["smoke_run"])
assert same_split, "Splits differ: results are not comparable"
print("model versions:", bl["model"]["version"], "/", ae["model"]["version"], "| input set:", ae["input"])
assert ae["model"]["version"] == autoencoder.MODEL_VERSION, "Report was produced by older code; re-run step 3"

hist = pd.DataFrame(ae["history"])
ax = hist.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(7, 3), title="Autoencoder loss (standardised MSE)")
ax.set_ylabel("loss"); plt.show()
print(f"epochs run: {len(hist)} | mean epoch time: {hist['seconds'].mean():.1f}s")

In [ ]:
metrics = ["precision", "recall", "f1", "fpr", "pr_auc", "roc_auc"]
rows = {
    "Isolation Forest": bl["primary_stratified"]["test_at_budget_by_subset"]["all"],
    "Autoencoder": ae["test_at_budget"]["all"],
    "Isolation Forest (without 657)": bl["primary_stratified"]["test_at_budget_by_subset"]["without_aircraft_657"],
    "Autoencoder (without 657)": ae["test_at_budget"]["without_aircraft_657"],
}
display(pd.DataFrame({k: {m: v["binary"][m] for m in metrics} for k, v in rows.items()}).T.round(3))
display(pd.DataFrame({k: v["alert_rate_by_class"] for k, v in rows.items()}).T.round(3)
        .rename(columns={"Nominal": "Nominal (false alarms)"}))

In [ ]:
if "aircraft_held_out" in ae:
    cv = pd.DataFrame({"Isolation Forest": {m: f"{s['mean']:.3f} ± {s['std']:.3f}" for m, s in bl["aircraft_held_out"]["summary"].items()},
                       "Autoencoder": {m: f"{s['mean']:.3f} ± {s['std']:.3f}" for m, s in ae["aircraft_held_out"]["summary"].items()}})
    display(cv)
print("Channels with the largest reconstruction-error increase vs nominal (supporting evidence, not causes):")
display(pd.DataFrame(ae["top_error_channels_by_class_test"]))

## 8. Package results

Unzip `flightops_results.zip` at the root of your local repo. It contains only `artifacts/` (models and reports), never data.

In [ ]:
bundle = REPO / "flightops_results.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted((REPO / "artifacts").rglob("*")):
        if p.is_file() and p.suffix in {".pt", ".joblib", ".json", ".npz", ".png"}:
            zf.write(p, p.relative_to(REPO))
print(f"{bundle} ({bundle.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(bundle))
elif IN_KAGGLE:
    shutil.copy2(bundle, "/kaggle/working/flightops_results.zip")
    print("Saved /kaggle/working/flightops_results.zip. Download it from the Output panel on the right "
          "(Output tab after 'Save Version'). Only this zip is in /kaggle/working; the dataset stays in /tmp.")